# پروژه پیش‌بینی قیمت خانه با CNN (Conv1D)

این Notebook برای مسابقه Kaggle **House Prices: Advanced Regression Techniques** آماده شده است.

**نکته:** مسئله رگرسیون است؛ بنابراین معیارهای اصلی MSE، RMSE، MAE و R² هستند. معیارهای Accuracy/Precision/Recall/F1 و ماتریس ابهام فقط به‌صورت تحلیل تکمیلی روی سه گروه قیمتی محاسبه می‌شوند. فایل test مسابقه ستون SalePrice ندارد؛ ارزیابی روی validation برگرفته از train انجام می‌شود.

ابتدا فایل‌های `train.csv` و `test.csv` را از صفحه رسمی مسابقه دانلود کنید و در پوشه `data/` قرار دهید. سپس سلول‌ها را به‌ترتیب اجرا کنید.


## ۱. نصب کتابخانه‌ها

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "train.py").exists() and (PROJECT_DIR.parent / "train.py").exists():
    PROJECT_DIR = PROJECT_DIR.parent
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")])
print("Installed project requirements from", PROJECT_DIR / "requirements.txt")


## ۲. تنظیم مسیرها و بارگذاری فایل‌ها در Google Colab (اختیاری)

اگر در Colab هستید، سلول بعدی را اجرا و هر دو فایل `train.csv` و `test.csv` را بارگذاری کنید. اگر فایل‌ها را قبلاً در پوشه data قرار داده‌اید، از این سلول صرف‌نظر کنید.

In [ ]:
from pathlib import Path
import os

# اگر Notebook درون پوشه notebooks اجرا شود، ریشه پروژه یک سطح بالاتر است.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "train.py").exists() and (PROJECT_DIR.parent / "train.py").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

# در Colab، در صورت نیاز دو فایل CSV را upload کنید:
try:
    from google.colab import files
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename.lower() in {"train.csv", "test.csv", "sample_submission.csv", "data_description.txt"}:
            (DATA_DIR / filename).write_bytes(content)
except ImportError:
    print("Google Colab شناسایی نشد؛ فایل‌ها را به‌صورت دستی در پوشه data قرار دهید.")


## ۳. کنترل فایل‌های ورودی

In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
assert train_path.exists(), f"فایل پیدا نشد: {train_path}"
assert test_path.exists(), f"فایل پیدا نشد: {test_path}"
print("هر دو فایل train.csv و test.csv پیدا شدند.")


## ۴. اجرای آموزش، ارزیابی و آزمایش معماری‌ها

با اجرای این سلول، مدل پایه و آزمایش‌های مقایسه‌ای آموزش داده می‌شوند. برای صرفه‌جویی در زمان می‌توانید `--skip-ablation` را به انتهای دستور اضافه کنید.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python train.py --train data/train.csv --test data/test.csv --output outputs --epochs 100 --batch-size 32


## ۵. مشاهده معیارهای اعتبارسنجی

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

OUT_DIR = PROJECT_DIR / "outputs"
metrics = json.loads((OUT_DIR / "evaluation_metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame([metrics]).T.rename(columns={0: "Validation value"}))


## ۶. نمودار MSE و خروجی‌های ارزیابی

In [ ]:
for filename in [
    "mse_by_epoch.png",
    "actual_vs_predicted.png",
    "residual_distribution.png",
    "confusion_matrix_price_bands.png",
    "ablation_rmse_log.png",
    "ablation_r2.png",
]:
    path = OUT_DIR / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))


## ۷. مشاهده جدول مقایسه معماری‌ها

In [ ]:
ablation_path = OUT_DIR / "ablation_results.csv"
if ablation_path.exists():
    display(pd.read_csv(ablation_path))
else:
    print("آزمایش مقایسه معماری‌ها اجرا نشده است.")


## ۸. خروجی آماده Kaggle

In [ ]:
submission = pd.read_csv(OUT_DIR / "submission.csv")
display(submission.head())
print("Shape:", submission.shape)
print("Submission file:", OUT_DIR / "submission.csv")


## ۹. دانلود خروجی‌ها در Google Colab (اختیاری)

In [ ]:
try:
    from google.colab import files
    files.download(str(OUT_DIR / "submission.csv"))
    files.download(str(OUT_DIR / "REPORT.md"))
except ImportError:
    print("برای محیط محلی، فایل‌ها در پوشه outputs قرار دارند.")
